Emre Alca
title: Steady-State and Transient Quantities tutorial.ipynb
date: 2025-01-17 21:20:30

In [12]:
from linearframework.linear_framework_graph import LinearFrameworkGraph
import linearframework.linear_framework_results as lfr
import sympy as sp

This notebook outlines the use of linearframework.py for the calculation of steady state probabilities, moments of first passage times, and splitting probabilities of LinearFrameworkGraph objects.

This note assumes familiarity with the material in `Graph Tutorial.ipynb`.

The closed form expressions for these quantities can be found in `linear-framework-ii.pdf` which is in this folder.

Let's recall our two graphs from Graph Tutorial.ipynb

In [2]:
k3_edges = [
    ('1', '2'),
    ('1', '3'),
    ('2', '1'),
    ('2', '3'),
    ('3', '1'),
    ('3', '2'),
]

k3 = LinearFrameworkGraph(k3_edges)

k3_terminal_edges = [
        ('1', '2'),
        ('1', '3'),
        ('2', '1'),
        ('2', '3'),
        ('3', '1'),
        ('3', '2'),
        ('2', '4'),
        ('3', '5')
    ]

k3_terminal = LinearFrameworkGraph(k3_terminal_edges)

Historically, the Linear Framework has been concerned with Markov Processes which have no terminal vertices, and particularly the analysis of their steady state probabilities. Recently methods have been developed which allow the analysis of transient quantities, such as the moments of the first passage time distributions and the splitting probabilities of graphs which do have terminal vertices.

Let's see how we can calculate all of these.

The formulas used to calculate the closed form expressions of these quantities can be found in this paper from the Gunawardena group:
[The linear framework II: using graph theory to analyse the transient regime of Markov processes](https://www.frontiersin.org/journals/cell-and-developmental-biology/articles/10.3389/fcell.2023.1233808/full)

We calculate all steady state probabilities simultaneously as a dictionary, where the vertices are the keys. That is `k3_steady_states['1']` is the expression for the steady state of being at state `'1'`.

In [4]:
k3_steady_states = lfr.steady_states_from_sym_lap(k3)

k3_steady_states['1']

(l_3*l_5 + l_3*l_6 + l_4*l_5)/(l_1*l_4 + l_1*l_5 + l_1*l_6 + l_2*l_3 + l_2*l_4 + l_2*l_6 + l_3*l_5 + l_3*l_6 + l_4*l_5)

Let's generate a random set of weights and use it to calculate a particular set of steady-states:

In [9]:
k3_edge_to_weight = k3.generate_random_edge_to_weight()

k3_sym_to_weight = k3.make_sym_to_weight(k3_edge_to_weight)

k3_steady_states['1'].subs(k3_sym_to_weight)

0.933108089677799

We can also calculate the expression for any arbitrarily high moment of the first-passage-time (FPT) distribution from some source vertex to some target vertex. Do note that this method is only intended for graphs which have no terminal vertices.

let's calculate the second moment of the FPT distribution on k3 from `'1'` to `'3'`.

In [7]:
k3_mth_moment_FPT = lfr.k_moment_fpt_expression(k3, '1', '3', 2)

k3_mth_moment_FPT

(2*l_1*l_3 + 2*l_1*(l_1 + l_2) + 2*l_1*(l_3 + l_4) + 2*(l_3 + l_4)**2)/(l_1*l_4 + l_2*l_3 + l_2*l_4)**2

And we can evaluate at a point.

In [10]:
k3_mth_moment_FPT.subs(k3_sym_to_weight)

19.4961552038077

We can also calculate the splitting probability of a system which has terminal vertices.

In [13]:
k3_terminal_splitting_probability = lfr.splitting_probability_ca(k3_terminal, '1', '5')

sp.simplify(k3_terminal_splitting_probability)

l_8*(l_1*l_4 + l_2*l_3 + l_2*l_4 + l_2*l_7)/(l_1*l_4*l_8 + l_1*l_5*l_7 + l_1*l_6*l_7 + l_1*l_7*l_8 + l_2*l_3*l_8 + l_2*l_4*l_8 + l_2*l_6*l_7 + l_2*l_7*l_8)

Which we can also evaluate at a point.

In [20]:
k3_terminal_edge_to_weight = k3_terminal.generate_random_edge_to_weight()

k3_terminal_sym_to_weight = k3_terminal.make_sym_to_weight(k3_terminal_edge_to_weight)

k3_terminal_splitting_probability.subs(k3_terminal_sym_to_weight)

0.700643302506116

A function for the calculation of the moments of the conditional FPT distribution is coming soon.